In [113]:
#
import os
import pickle
import numpy as np
import pandas as pd
import torchaudio
from pathlib import Path

df_annotations = pd.read_excel(f'../data/annotations_2.xlsx')
df_reduced = df_annotations.drop(columns=['Name', 'Emotion', 'Annotator'])
df_reduced = df_reduced.drop_duplicates()

In [133]:
seven = []
six = []
emotions = ['Valence', 'Arousal', 'Dominance']
for row in df_reduced.sample(frac=1).reset_index(drop=True).iterrows():
    row = row[1]
    for emo in emotions:
        df = df_annotations[(df_annotations.PC_Num == row['PC_Num']) & (df_annotations.Part_Num == row['Part_Num']) & (df_annotations.Emotion == emo)]
        if df.shape[0] > 6:
            seven.append(df.shape[0])
        else:
            six.append(df.shape[0])


In [135]:
# 121 annotations pueden ser mas de > 6, lo que trae error al seleccionar el idx # 6 annotations[6]
# lo cambie a annotations[-1]
len(seven), len(six)

In [138]:
from collections import Counter

Counter(seven), Counter(six)

In [105]:
pc_num, part_num, emotion = 760, 2, "Valence"
df_annotations[(df_annotations.PC_Num == pc_num) & (df_annotations.Part_Num == part_num) & (df_annotations.Emotion == emotion)]

,Name,Emotion,Annotator,PC_Num,Part_Num,start_time,end_time,Audio_Name,Type
2074,MSP-Conversation_0760_2_002.csv,Valence,2,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2075,MSP-Conversation_0760_2_005.csv,Valence,5,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2076,MSP-Conversation_0760_2_006.csv,Valence,6,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2077,MSP-Conversation_0760_2_007.csv,Valence,7,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2078,MSP-Conversation_0760_2_008.csv,Valence,8,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2079,MSP-Conversation_0760_2_009.csv,Valence,9,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train
2080,MSP-Conversation_0760_2_011.csv,Valence,11,760,2,339.921,601.2406,MSP-Conversation_0760.wav,Train


In [ ]:
import os
import pickle
import numpy as np
import torchaudio
from pathlib import Path
from vad.vad_lab import VAD

input_pkl = Path(os.getcwd()).parent / "data/input_features-train.pkl"
input_fixed_pkl = Path(os.getcwd()).parent / "data/input_features-train-fixed.pkl"

input_features = {}

try:
    with open(input_pkl, "rb") as pkl:
        input_features = pickle.load(pkl)
except Exception as ex:
    print(ex)



In [ ]:
def fix_input_features(input_features):
    durations = []
    fixed_input_features = {}
    inputs, labels = [], []
    stats = []
    for i, (input, label) in enumerate(zip(input_features['inputs'], input_features['labels'])):
        path = Path(os.getcwd()).parent / "audiosegments" / input
        wave, sr = torchaudio.load(path, normalize=True)
        duration = wave.shape[1] / sr
        mean = wave[0].mean().item()
        std = wave[0].std().item()
        if duration >= 3 and duration <= 6:
            if "1568_1_4_66.114_70.014_Train" in input:
                print(duration, input)
            aud = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/{input}"
            aud_dB = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/dBDown/{input}"
            wave, sr = torchaudio.load(aud, normalize=True)
            wave = wave.squeeze() # [c, s] => [s]
            wave = wave * 0.25
            torchaudio.save(aud_dB, wave.unsqueeze(dim=0), sample_rate=16_000, bits_per_sample=16, encoding='PCM_S')
            
            durations.append(duration)
            stats.append([mean, std])
            inputs.append(input)
            labels.append(label)
    fixed_input_features['inputs'] = inputs
    fixed_input_features['labels'] = labels
    return np.array(durations), fixed_input_features, stats

durations, fixed_input_features,  stats = fix_input_features(input_features)
oldf = len(input_features['inputs'])
newf = len(durations)
print(f"extranger length of input_features: {oldf}, fixed length input features: {newf}")

In [ ]:
stats = np.array(stats)

In [ ]:
stats[:, 1]

In [ ]:
stats[0:3, :]

In [ ]:
import matplotlib.pyplot as plt
plt.hist(stats[:, 0], bins='auto')

In [ ]:

with open(input_fixed_pkl, "wb") as pkl:
    pickle.dump(fixed_input_features, pkl)
    

In [ ]:
import matplotlib.pyplot as plt
plt.hist(durations, bins='auto')

In [ ]:
vad = VAD(minmax=[-100, 100], mapping="OCC")
vad.terms
medium = {"aa": 1, "ae": 2, "ah": 3, "aw": 4, "ay": 5, "b": 6, "ch": 7, "d": 8, "dh": 9, "dx": 10, "eh": 11, "er": 12, "ey": 13, "f": 14, "g": 15, "h#": 16, "hh": 17, "ih": 18, "iy": 19, "jh": 20, "k": 21, "l": 22, "m": 23, "ng": 24, "|": 0, "[UNK]": 25, "[PAD]": 26}

all_labels = []
for label in fixed_input_features['labels']:
    all_labels.extend(label.split())

ids_labels = []
for l in all_labels:
    ids_labels.append(medium[l])

emos_labels = []
for i in ids_labels:
    emos_labels.append(vad.terms[i-1])



In [ ]:
np.vstack((ids_labels, emos_labels))

In [ ]:
import pandas as pd

ids_labels
ids_labels_df = pd.DataFrame(np.vstack((ids_labels, emos_labels))).T
ids_labels_df[1].hist()
plt.xticks(rotation='vertical')


In [ ]:
Path(os.getcwd()).parent / 'data/wadf.pkl'

In [5]:
import os
import numpy as np
from pathlib import Path
import pickle

# mspconvs = {}
# try:
#     with open(Path(os.getcwd()).parent / 'data/mspconvs.pkl', "rb") as pkl:
#         mspconvs = pickle.load(pkl)
# except Exception as ex:
#     print(ex)

wadf = {}
try:
    with open(Path(os.getcwd()).parent / 'data/wadf.pkl', "rb") as pkl:
        wadf = pickle.load(pkl)
except Exception as ex:
    print(ex)

In [11]:
from vad.vad_lab import VAD
vad = VAD(minmax=[-100, 100], mapping="OCC")


r = "184_1_51_313.602_317.31_Test_REPROACH_-15.802_27.0411_43.7082".split("_")
part = f"{r[0]}_{r[1]}"
start, end = float(r[3]), float(r[4])

wa_valence = wadf[f'{part}_Valence']

wa_arousal = wadf[f'{part}_Arousal']

wa_dominance = wadf[f'{part}_Dominance']


overlap = 0.
wa = wa_valence[(wa_valence['Time'] > (start - overlap)) & (wa_valence['Time'] < (end + overlap))]
v = np.nan_to_num(wa['Annotation'].to_numpy()).mean()

wa = wa_arousal[(wa_arousal['Time'] > (start - overlap)) & (wa_arousal['Time'] < (end + overlap))]
a = np.nan_to_num(wa['Annotation'].to_numpy()).mean()

wa = wa_dominance[(wa_dominance['Time'] > (start - overlap)) & (wa_dominance['Time'] < (end + overlap))]
d = np.nan_to_num(wa['Annotation'].to_numpy()).mean()

print(v,a,d)

vad.vad2categorical(v,a,d,k=3)
# wa_valence

-15.871473262511996 27.032152415596357 43.76464626893987


([{'term': 'REPROACH',
   'index': 20,
   'closest': 0.2244731710495179,
   'v': -0.3,
   'a': 0.1,
   'd': 0.4},
  {'term': 'RESENTMENT',
   'index': 21,
   'closest': 0.24302495028170293,
   'v': -0.2,
   'a': 0.3,
   'd': 0.2},
  {'term': 'GLOATING',
   'index': 7,
   'closest': 0.36721583908143174,
   'v': -0.3,
   'a': 0.3,
   'd': 0.1}],
 {'using_dominance': True})

In [ ]:
wadf[f'{part}_Dominance']

In [ ]:
wadf[f'{part}_Arousal']

In [ ]:
mspconvs['2252_2_Arousal'].keys()

In [ ]:
mspconvs['1140_3_Dominance']['reps_scaled'][:, 0]

In [ ]:
mspconvs['1140_3_Dominance']['annolist']

In [ ]:
mspconvs['2_1_Arousal']['annotations'][6]

### TIMIT AREA


In [ ]:
def read_text_file(filepath):
    with open(filepath) as f:
        tokens = [line.split()[-1] for line in f]
        return " ".join(tokens)

f = '/Users/beltre.wilton/apps/msp_temp/TIMIT/all_txt/SX308.PHN'
read_text_file(f)

In [ ]:
from pathlib import Path

inputs = []
labels = []
for file in Path('/Users/beltre.wilton/apps/msp_temp/TIMIT/all_waves/train').rglob("*"):
    inputs.append(str(file))
    phn = f'/Users/beltre.wilton/apps/msp_temp/TIMIT/all_txt/train/{file.name.replace(".WAV.wav", ".PHN")}'
    label = read_text_file(phn)
    labels.append(label)



In [ ]:
timit_input_features_train = {}
timit_input_features_train['inputs'] = inputs
timit_input_features_train['labels'] = labels

In [ ]:
def __save_input_features(dataset_object, name):
    with open(f'/Users/beltre.wilton/apps/mspconv_ftlab/data/{name}', "wb") as pkl:
        pickle.dump(dataset_object, pkl)

__save_input_features(timit_input_features_train, "timit_input_features-train.pkl")

## Classification Area

In [ ]:
import os
import pickle
import numpy as np
import torchaudio
from pathlib import Path
from vad.vad_lab import VAD

input_pkl = Path(os.getcwd()).parent / "data/class_input_features-test.pkl"
input_fixed_pkl = Path(os.getcwd()).parent / "data/class_input_features-test-fixed.pkl"

input_features = {}

try:
    with open(input_pkl, "rb") as pkl:
        input_features = pickle.load(pkl)
except Exception as ex:
    print(ex)

vad = VAD(minmax=[-100, 100], mapping="OCC")


In [ ]:
inputs = []
labels = []
class_input_features_test_fixed = {}
for input, label in zip(input_features['inputs'], input_features['labels']):
    inputs.append(input)
    labels.append(vad.terms[label[0]-1])

class_input_features_test_fixed['inputs'] = inputs
class_input_features_test_fixed['labels'] = labels


In [ ]:
vad.vad2categorical(0.4, 0.6, -1, k=1, use_plot=False)

In [ ]:
def __save_input_features(dataset_object, name):
    with open(f'/Users/beltre.wilton/apps/mspconv_ftlab/data/{name}', "wb") as pkl:
        pickle.dump(dataset_object, pkl)

__save_input_features(class_input_features_test_fixed, "class_input_features_test_fixed.pkl")

In [ ]:
vad.terms

In [ ]:
# print("14: ", vad.terms[14])

# for label in class_input_features_test_fixed['labels']:
#     print(label, list(vad.terms).index(label))


class_input_features_test_fixed['inputs']

## Randomized Data Approach

In [2]:
import pickle

class_input_features_development_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_development_robust_20240828.pkl"
class_input_features_test_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_test_robust_20240828.pkl"
class_input_features_train_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_train_robust_20240828.pkl"

input_features = {}

def load_inputs(input_pkl):
  try:
      with open(input_pkl, "rb") as pkl:
          input_features = pickle.load(pkl)
          return input_features
  except Exception as ex:
      print(ex)


train_input_features = load_inputs(class_input_features_train_fixed)
test_input_features = load_inputs(class_input_features_test_fixed)
dev_input_features = load_inputs(class_input_features_development_fixed)


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets


output1 = widgets.Output()
with output1:
    ids_labels_df_train = pd.DataFrame(train_input_features['labels'])
    print("Train\n--------------------------")
    display(ids_labels_df_train.value_counts().sort_index(ascending=False))

output2 = widgets.Output()
with output2:
    print("Eval\n--------------------------")
    ids_labels_df_dev = pd.DataFrame(dev_input_features['labels'])
    display(ids_labels_df_dev.value_counts().sort_index(ascending=False))

output3 = widgets.Output()
with output3:
    print("Test\n--------------------------")
    ids_labels_df_test = pd.DataFrame(test_input_features['labels'])
    display(ids_labels_df_test.value_counts().sort_index(ascending=False))


columns = widgets.HBox([output1, output2, output3])
display(columns)

#TODO talvez re-distribuir en las catagorias "mas fuertes" 


In [69]:
π

old input_features: 3385, fixed length input features: 2207
old input_features: 4783, fixed length input features: 3109
old input_features: 12211, fixed length input features: 7949


In [72]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets


output1 = widgets.Output()
with output1:
    ids_labels_df_train = pd.DataFrame(train_input_features['labels'])
    print("Train\n--------------------------")
    display(ids_labels_df_train.value_counts().sort_index(ascending=False))

output2 = widgets.Output()
with output2:
    print("Eval\n--------------------------")
    ids_labels_df_dev = pd.DataFrame(dev_input_features['labels'])
    display(ids_labels_df_dev.value_counts().sort_index(ascending=False))

output3 = widgets.Output()
with output3:
    print("Test\n--------------------------")
    ids_labels_df_test = pd.DataFrame(test_input_features['labels'])
    display(ids_labels_df_test.value_counts().sort_index(ascending=False))


columns = widgets.HBox([output1, output2, output3])
display(columns)


In [71]:
import random

def down_sampler(input_features, emos, factor=(320, 420)):
    emos_dict = {}
    for input, label in zip(input_features['inputs'], input_features['labels']):
        if len(emos_dict.get(label, [])) == 0:
            emos_dict[label] = []
            emos_dict[label].append(input)
        else:
            emos_dict[label].append(input)

    inputs = []
    labels = []
    # filter by emo
    for emo in emos:
        ranint = random.randint(*factor)
        current = emos_dict[str(emo)]
        if len(current)  < 20:
            continue
        samples = random.sample(current, ranint) if len(current) > ranint else current
        for input, label in zip(input_features['inputs'], input_features['labels']):
            if input in samples:
                inputs.append(input)
                labels.append(label)
            
            

    return {"inputs": inputs, "labels": labels}


emos = [i[0] for i in ids_labels_df_train.value_counts().index]
train_input_features = down_sampler(train_input_features, emos)

emos = [i[0] for i in ids_labels_df_dev.value_counts().index]
dev_input_features = down_sampler(dev_input_features, emos, factor=(90, 100))

emos = [i[0] for i in ids_labels_df_test.value_counts().index]
test_input_features = down_sampler(test_input_features, emos, factor=(90, 100))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def merge_features(all_features):
    inputs = []
    labels = []
    input_features_merged = {}
    for input_features in all_features:
        for input, label in zip(input_features['inputs'], input_features['labels']):
            inputs.append(input)
            labels.append(label)
        input_features_merged['inputs'] = inputs
        input_features_merged['labels'] = labels
    
    return input_features_merged


all_features = [train_input_features, test_input_features, dev_input_features]
input_features_merged = merge_features(all_features)

In [ ]:
len(train_input_features['inputs']), len(input_features_merged['inputs'])

In [ ]:
df = pd.DataFrame(input_features_merged)
X_train, X_eval = train_test_split(df, train_size = 0.60)
X_eval, X_test = train_test_split(X_eval, train_size = 0.4)
X_train.shape[0], X_eval.shape[0], X_test.shape[0]

In [ ]:
len(train_input_features['inputs']), len(dev_input_features['inputs']), len(test_input_features['inputs'])

In [ ]:
train_input_features =  X_train.to_dict("list")
dev_input_features =  X_eval.to_dict("list")
test_input_features =  X_test.to_dict("list")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets


output1 = widgets.Output()
with output1:
    ids_labels_df_train = pd.DataFrame(train_input_features['labels'])
    print("Train\n--------------------------")
    display(ids_labels_df_train.value_counts().sort_index(ascending=False))

output2 = widgets.Output()
with output2:
    print("Eval\n--------------------------")
    ids_labels_df_dev = pd.DataFrame(dev_input_features['labels'])
    display(ids_labels_df_dev.value_counts().sort_index(ascending=False))

output3 = widgets.Output()
with output3:
    print("Test\n--------------------------")
    ids_labels_df_test = pd.DataFrame(test_input_features['labels'])
    display(ids_labels_df_test.value_counts().sort_index(ascending=False))


columns = widgets.HBox([output1, output2, output3])
display(columns)


In [ ]:
# sanity check
import torchaudio


for i, (input, label) in enumerate(zip(train_input_features['inputs'], train_input_features['labels'])):
        path = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/{input}"
        wave, sr = torchaudio.load(path, normalize=True)
        duration = wave.shape[1] / sr
        if duration > 4.5:
            print(path, duration)

In [73]:
def __save_input_features(dataset_object, name):
    with open(f'/Users/beltre.wilton/apps/mspconv_ftlab/data/{name}', "wb") as pkl:
        pickle.dump(dataset_object, pkl)


__save_input_features(train_input_features, "class_input_features_train_robust_20240827_.pkl")
__save_input_features(dev_input_features, "class_input_features_development_robust_20240827_.pkl")
__save_input_features(test_input_features, "class_input_features_test_robust_20240827_.pkl")

## Clean data for classification

In [ ]:
def __save_input_features(dataset_object, name):
    with open(f'{name}', "wb") as pkl:
        pickle.dump(dataset_object, pkl)

In [61]:
import pickle

class_input_features_development_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_development_robust_20240827.pkl"
class_input_features_test_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_test_robust_20240827.pkl"
class_input_features_train_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_train_robust_20240827.pkl"

# class_input_features_development_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features-development.pkl"
# class_input_features_test_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features-test.pkl"
# class_input_features_train_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features-train.pkl"

input_features = {}

def load_inputs(input_pkl):
  try:
      with open(input_pkl, "rb") as pkl:
          input_features = pickle.load(pkl)
          return input_features
  except Exception as ex:
      print(ex)


train_input_features = load_inputs(class_input_features_train_fixed)
test_input_features = load_inputs(class_input_features_test_fixed)
dev_input_features = load_inputs(class_input_features_development_fixed)

In [62]:
import os
import numpy as np
import torchaudio
import pickle
from pathlib import Path

# class_input_features_development_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_development_fixed.pkl"
# class_input_features_test_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_test_fixed.pkl"
# class_input_features_train_fixed = "/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_train_fixed.pkl"


# input_features = {}

# def load_inputs(input_pkl):
#   try:
#       with open(input_pkl, "rb") as pkl:
#           input_features = pickle.load(pkl)
#           return input_features
#   except Exception as ex:
#       print(ex)
    

# input_features_development = load_inputs(class_input_features_development_fixed)
# input_features_test = load_inputs(class_input_features_test_fixed)
# input_features_train = load_inputs(class_input_features_train_fixed)


def fix_input_features(input_features):
    durations = []
    fixed_input_features = {}
    inputs, labels = [], []
    stats = []
    for i, (input, label) in enumerate(zip(input_features['inputs'], input_features['labels'])):
        path = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/{input}"
        wave, sr = torchaudio.load(path, normalize=True)
        duration = wave.shape[1] / sr
        mean = wave[0].mean().item()
        std = wave[0].std().item()
        if duration >= 2.5 and duration <= 4.5:
            durations.append(duration)
            stats.append([mean, std])
            inputs.append(input)
            labels.append(label)
    fixed_input_features['inputs'] = inputs
    fixed_input_features['labels'] = labels
    return np.array(durations), fixed_input_features, stats

oldf = len(dev_input_features['inputs'])
durations, dev_input_features,  stats = fix_input_features(dev_input_features)
newf = len(durations)
print(f"old input_features: {oldf}, fixed length input features: {newf}")

oldf = len(test_input_features['inputs'])
durations, test_input_features,  stats = fix_input_features(test_input_features)
newf = len(durations)
print(f"old input_features: {oldf}, fixed length input features: {newf}")

oldf = len(train_input_features['inputs'])
durations, train_input_features,  stats = fix_input_features(train_input_features)
newf = len(durations)
print(f"old input_features: {oldf}, fixed length input features: {newf}")


old input_features: 3385, fixed length input features: 2207
old input_features: 4783, fixed length input features: 3109
old input_features: 12211, fixed length input features: 7949


In [ ]:
np.mean([std[0] for std in stats]), np.mean([std[1] for std in stats])

In [ ]:
from vad.vad_lab import VAD

vad = VAD(minmax=[-100, 100], mapping="OCC")

def ids2cat(input_features):
    inputs = []
    labels = []
    input_features_fixed = {}
    for input, label in zip(input_features['inputs'], input_features['labels']):
        inputs.append(input)
        labels.append(vad.terms[label[0]-1])

    input_features_fixed['inputs'] = inputs
    input_features_fixed['labels'] = labels
    
    return input_features_fixed


train_input_features = ids2cat(input_features_train)
test_input_features = ids2cat(input_features_test)
dev_input_features = ids2cat(input_features_development)


In [ ]:
train_input_features['inputs'][:5], train_input_features['labels'][:5]

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets



output1 = widgets.Output()
with output1:
    ids_labels_df_train = pd.DataFrame(train_input_features['labels'])
    print("Train\n--------------------------")
    display(ids_labels_df_train.value_counts().sort_index(ascending=False))

output2 = widgets.Output()
with output2:
    print("Eval\n--------------------------")
    ids_labels_df_dev = pd.DataFrame(dev_input_features['labels'])
    display(ids_labels_df_dev.value_counts().sort_index(ascending=False))

output3 = widgets.Output()
with output3:
    print("Test\n--------------------------")
    ids_labels_df_test = pd.DataFrame(test_input_features['labels'])
    display(ids_labels_df_test.value_counts().sort_index(ascending=False))


columns = widgets.HBox([output1, output2, output3])
display(columns)

# [0 'RESENTMENT', 1 'SATISFACTION', 2 'REPROACH', 3 'LOVE', 4 'NEUTRAL', 5 'PRIDE', 6'ANGER', 
#  7 'HOPE', 8 'GLOATING', 9 'PITY', 10 'HATE',  11 'HAPPY FOR?', 12 'DISLIKING', 13 'JOY']

In [65]:
import random

def down_sampler(input_features, emos, factor=(320, 420)):
    emos_dict = {}
    for input, label in zip(input_features['inputs'], input_features['labels']):
        if len(emos_dict.get(label, [])) == 0:
            emos_dict[label] = []
            emos_dict[label].append(input)
        else:
            emos_dict[label].append(input)

    inputs = []
    labels = []
    # filter by emo
    for emo in emos:
        ranint = random.randint(*factor)
        current = emos_dict[str(emo)]
        if len(current)  < 20:
            continue
        samples = random.sample(current, ranint) if len(current) > ranint else current
        for input, label in zip(input_features['inputs'], input_features['labels']):
            if input in samples:
                inputs.append(input)
                labels.append(label)
            
            

    return {"inputs": inputs, "labels": labels}


emos = [i[0] for i in ids_labels_df_train.value_counts().index]
train_input_features = down_sampler(train_input_features, emos)

emos = [i[0] for i in ids_labels_df_dev.value_counts().index]
dev_input_features = down_sampler(dev_input_features, emos, factor=(100, 120))

emos = [i[0] for i in ids_labels_df_test.value_counts().index]
test_input_features = down_sampler(test_input_features, emos, factor=(100, 350))

In [66]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets



output1 = widgets.Output()
with output1:
    ids_labels_df_train = pd.DataFrame(train_input_features['labels'])
    print("Train\n--------------------------")
    display(ids_labels_df_train.value_counts().sort_index(ascending=False))

output2 = widgets.Output()
with output2:
    print("Eval\n--------------------------")
    ids_labels_df_dev = pd.DataFrame(dev_input_features['labels'])
    display(ids_labels_df_dev.value_counts().sort_index(ascending=False))

output3 = widgets.Output()
with output3:
    print("Test\n--------------------------")
    ids_labels_df_test = pd.DataFrame(test_input_features['labels'])
    display(ids_labels_df_test.value_counts().sort_index(ascending=False))


columns = widgets.HBox([output1, output2, output3])
display(columns)
terms = ['RESENTMENT', 'SATISFACTION', 'REPROACH', 'LOVE', 'NEUTRAL', 'PRIDE', 'ANGER', 'HOPE', 'GLOATING', 'PITY', 'HATE', 'HAPPY FOR?', 'DISLIKING', 'JOY']

In [ ]:
len(train_input_features['labels'])

In [ ]:
def __save_input_features(dataset_object, name):
    with open(f'/Users/beltre.wilton/apps/mspconv_ftlab/data/{name}', "wb") as pkl:
        pickle.dump(dataset_object, pkl)


__save_input_features(train_input_features, "class_input_features_train_robust.pkl")
# __save_input_features(dev_input_features, "class_input_features_development_robust.pkl")
# __save_input_features(test_input_features, "class_input_features_test_robust.pkl")

In [ ]:
# sanity check
for i, (input, label) in enumerate(zip(train_input_features['inputs'], train_input_features['labels'])):
        path = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/{input}"
        wave, sr = torchaudio.load(path, normalize=True)
        duration = wave.shape[1] / sr
        if duration > 5.5:
            print(path, duration)

            

In [ ]:
[[i+1, term]for i, term in enumerate(vad.terms)]

In [ ]:
dev = ids_labels_df_train.value_counts().to_frame()
dev['indx'] = dev.index.to_series().apply(lambda x: list(vad.terms).index(x[0]) + 1)
dev

In [ ]:
def plot_audio(wave, path_file, emo):
    display(Audio(data=wave, autoplay=False, rate=16_000, ))
    # Generate some sample data (e.g., a sine wave)
    t = np.linspace(0, len(wave.squeeze()) / 16_000, len(wave.squeeze()))  # Time points
    # waveform = np.sin(t)  # Sine wave
    
    # Plot the waveform
    plt.figure(figsize=(16, 1))  # Set figure size
    plt.plot(t, wave, color='blue', alpha=.5, ms=10)  # Plot waveform
    plt.title(f'{emo} {path_file}')  # Set title
    plt.xlabel('Time')  # Set x-axis label
    plt.ylabel('Amplitude')  # Set y-axis label
    plt.grid(True)  # Add grid
    plt.show()  # Show plot



In [ ]:
import torch
import torchaudio

import random
from pathlib import Path

knn_vc = torch.hub.load('bshall/knn-vc', 'knn_vc', prematched=True, trust_repo=True, pretrained=True, device='cpu')


def get_mess_reference():
    mess16k = Path("/Users/beltre.wilton/apps/ftlab_w2v2_ser/rawdata/mess_16k/")
    mess16klist = [file.name for file in mess16k.rglob("*")]
    pick = random.sample(mess16klist, 1)[0][:4]
    return random.sample([str(mess16k / mess) for mess in mess16klist if pick in mess], 1)


def _knnvc(in_wave_file: str, audio_ref: [], emo: str, plot=False):
    # path to 16kHz, single-channel, source waveform
    # src_wav_path = '/content/D.wav'
    # list of paths to all reference waveforms (each must be 16kHz, single-channel) from the target speaker
    # ref_wav_paths = ['/content/D_s.wav', ]

    in_wave, sr = torchaudio.load(in_wave_file) # normalize=True
    refer = []
    for idx, ref in enumerate(audio_ref):
        r, _ = torchaudio.load(ref) # normalize=True
        refer.append(r.squeeze().numpy())
        if plot:
            plot_audio(r.squeeze().numpy(), ref, f"reference#{idx+1}")

    in_wave = torch.from_numpy(in_wave.squeeze().numpy())
    refer = torch.from_numpy(np.array(refer))

    if plot:
        plot_audio(in_wave, in_wave_file, emo)
    
    query_seq = knn_vc.get_features(in_wave)
    matching_set = knn_vc.get_matching_set([refer])

    out_wav = knn_vc.match(query_seq, matching_set, topk=4)

    synth_out_wave_file = f'{in_wave_file.replace(".wav", "_synth.wav")}'

    torchaudio.save(synth_out_wave_file, out_wav[None], sample_rate=16_000, bits_per_sample=16, encoding='PCM_S')

    if plot:
        plot_audio(out_wav, synth_out_wave_file, "SYNTH FILE: ")
    
    return synth_out_wave_file


In [ ]:
emos_dict = {}
for input, label in zip(dev_input_features['inputs'], dev_input_features['labels']):
    if len(emos_dict.get(label, [])) == 0:
        emos_dict[label] = []
        emos_dict[label].append(input)
    else:
        emos_dict[label].append(input)

In [ ]:
import matplotlib.pyplot as plt


# train_input_features = synth_input_features 
ids_labels_df = pd.DataFrame(dev_input_features['labels'])
ids_labels_df.value_counts().plot( kind="barh")

# ids_labels_df = pd.DataFrame(test_input_features['labels'])
# ids_labels_df.value_counts().plot(kind="barh")

# ids_labels_df = pd.DataFrame(train_input_features['labels'])
# ids_labels_df.value_counts().plot(kind="barh")
ids_labels_df.value_counts()

In [ ]:
import random
import torchaudio
from IPython.display import display, Audio
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm import tqdm



N_SYNTH = {"HAPPY FOR?": 64, "JOY": 75  }
synth_input_features = {}
inputs = []
labels = []
for emo, N in N_SYNTH.items():
    for n in tqdm(range(N), desc=f"sintetizando {N} {emo}  ⏳...", total=N, ncols=120):
        elements = random.sample(emos_dict[emo], 1)
        for elem in elements:
            audio = f"/Users/beltre.wilton/apps/mspconv_ftlab/audiosegments/{elem}"
            try:
                synth_out_wave_file = _knnvc(audio, get_mess_reference(), emo)
                inputs.append(str(Path(synth_out_wave_file).name))
                labels.append(emo)
            except Exception as ex:
                continue

synth_input_features['inputs'] = inputs
synth_input_features['labels'] = labels


In [ ]:
len(synth_input_features['inputs'])

In [ ]:
# inputs = []
# labels = []
# for input, label in zip(synth_input_features['inputs'], synth_input_features['labels']):
#     dev_input_features['inputs'].append(input)
#     dev_input_features['labels'].append(label)


# inputs = []
# labels = []
# dropped = {}
# for input, label in zip(train_input_features['inputs'], train_input_features['labels']):
#     if label not in ('LIKING', 'DISAPPOINTMENT', 'GRATITUDE'):
#         inputs.append(input)
#         labels.append(label)

# dropped['inputs'] = inputs
# dropped['labels'] = labels
# train_input_features = dropped


with open('/Users/beltre.wilton/apps/mspconv_ftlab/data/class_input_features_development_robust.pkl', "wb") as pkl:
    pickle.dump(dev_input_features, pkl)


In [ ]:
import matplotlib.pyplot as plt


ids_labels_df = pd.DataFrame(train_input_features['labels'])
ids_labels_df.value_counts().plot( kind="barh")
ids_labels_df.value_counts()

## Mean Vote Area

In [ ]:
import json

targets_mean_vote = {}
with open('../data/targets_mean_vote.json', "r") as tgt:
    targets_mean_vote = json.load(tgt)

In [ ]:
for tgt in targets_mean_vote.items():
    print(tgt[0], len(tgt[1]['rangos']))

In [ ]:
from IPython.display import display, Audio
import numpy as np
import random
import matplotlib.pyplot as plt
import numpy as np
import torchaudio

wave, sr = torchaudio.load('/Users/beltre.wilton/Downloads/SER-Datasets/MSP-Conversation-1.1/Audio/MSP-Conversation_0690.wav')

# Generate some sample data (e.g., a sine wave)
t = np.linspace(0, round(len(wave[0]) / 16_000) , round(len(wave[0]) / 16_000))  # Time points
waveform = np.sin(t)  # Sine wave

# Plot the waveform
plt.figure(figsize=(16, 1))  # Set figure size
plt.plot(t, wave[0][::], color='blue')  # Plot waveform
plt.title('Waveform Plot')  # Set title
plt.xlabel('Time')  # Set x-axis label
plt.ylabel('Amplitude')  # Set y-axis label
plt.grid(True)  # Add grid
plt.show()  # Show plot

In [ ]:
print(targets_mean_vote['MSP-Conversation_0678.wav']['rangos'])


In [ ]:
 try:
    with open(path_file, "rb") as pkl:
        input_features = pickle.load(pkl)
        print(f"inputs:{len(input_features['inputs'])}, labels:{len(input_features['labels'])}")
except Exception as ex:
    print(ex)

## Plot area

In [ ]:
from vad.vad_lab import VAD

vad = VAD(minmax=[-100, 100], mapping="OCC")

In [ ]:
vad.vad2categorical(6.9125, 18.0019, 28.9395, k=3, use_plot=True)

In [ ]:
vad.plot(title="361_3_13_121.41_127.39_Train_SATISFACTION_6.9125_18.0019_28.9395")